# 04. RAG (Retrieval-Augmented Generation)

## 학습 목표
- RAG의 동기와 LLM의 지식 한계 이해
- 문서 청킹 전략 실습 (고정 크기, 문장 단위, 의미 단위)
- Embedding 모델과 벡터 DB (FAISS) 사용법
- 전체 RAG 파이프라인 구현
- RAG 평가 기초 (Faithfulness, Relevancy)

## 핵심 논문
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) (Lewis et al., 2020)

---

In [ ]:
# 필요한 라이브러리 설치 (Colab에서 실행)
# !pip install -q faiss-cpu sentence-transformers

import json
import re
import textwrap
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

# FAISS 및 sentence-transformers는 선택적
try:
    import faiss
    FAISS_AVAILABLE = True
    print("FAISS 사용 가능")
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS 미설치 - NumPy 기반 벡터 검색으로 대체")
    print("\uc124\uce58: !pip install faiss-cpu")

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
    print("SentenceTransformers 사용 가능")
except ImportError:
    SBERT_AVAILABLE = False
    print("SentenceTransformers 미설치 - 랜덤 임베딩으로 대체")
    print("\uc124\uce58: !pip install sentence-transformers")

## 1. RAG의 동기: LLM의 지식 한계

### LLM의 한계

| 한계 | 설명 | 예시 |
|------|------|------|
| **지식 단절** | 학습 데이터 이후 정보 모름 | "2025년 최신 뉴스는?" |
| **Hallucination** | 모르는 것을 자신 있게 지어냄 | 없는 매장을 추천 |
| **도메인 지식** | 특정 조직/서비스 내부 정보 모름 | "우리 회사 휴가 정책은?" |

### RAG의 해결 방법

```
\uc0ac\uc6a9\uc790 \uc9c8\ubb38 \u2192 \uad00\ub828 \ubb38\uc11c \uac80\uc0c9 \u2192 \ubb38\uc11c + \uc9c8\ubb38\uc744 LLM\uc5d0 \uc804\ub2ec \u2192 \uadfc\uac70 \uc788\ub294 \ub2f5\ubcc0
```

RAG = **Retrieval** (\uac80\uc0c9) + **Augmented** (\uc99d\uac15) + **Generation** (\uc0dd\uc131)

---
## 2. RAG 파이프라인 개요

```
문서 준비 (Offline):
  문서 → 청킹(Chunking) → 임베딩(Embedding) → 벡터 DB 저장

쿼리 처리 (Online):
  질문 → 임베딩 → 벡터 DB 검색 → 상위 k개 청크 → LLM에 전달 → 답변 생성
```

In [ ]:
# --- 샘플 문서: 한국 커피 문화 위키피디아 (가상) ---

documents = [
    {
        "title": "한국의 커피 문화",
        "content": """한국의 커피 문화는 1990년대 후반부터 급속히 발전하였다. 
1999년 스타벅스가 한국에 첫 매장을 열었으며, 이후 한국의 커피 시장은 폭발적으로 성장하였다.
2020년 기준 한국의 커피 시장 규모는 약 7조 원으로, 세계에서 손꼽힘는 커피 소비국이다.
한국인의 1인당 커피 소비량은 연간 약 353잔으로, 세계 평균을 크게 상회한다.
특히 스페셜티 커피와 드립 커피의 인기가 높아, 숨은 로스터리와 커피 전문점이 급증하였다."""
    },
    {
        "title": "서울의 카페 거리",
        "content": """서울에는 다양한 카페 거리가 있다. 성수동은 트렌디한 카페와 로스터리가 밀집해 있는 지역으로,
블루보틀, 컨테이너 등 유명 카페가 위치해 있다. 홍대입구역 주변은 개성 있는
독립 카페가 많으며, 연남동과 연희동 일대는 고즐닝한 분위기의 카페가 인기다.
성수동의 블루보틀은 2019년에 개점하여 스페셜티 드립 커피로 유명해졌다.
이태원에는 그린빈스 커피를 시작으로 다양한 카페가 형성되어 있다."""
    },
    {
        "title": "커피 드립 방법",
        "content": """커피 드립 방법에는 여러 가지가 있다. 핸드드립은 뜨거운 물을 커피 가루 위에 부어
천천히 내리는 방법으로, 깨끗한 맛을 낸다. 에스프레소는 고압으로 커피를 추출하는 방법으로,
농축된 맛과 크레마가 특징이다. 콜드브루는 차가운 물로 장시간 우려내는 방법으로,
부드럽고 달콤한 맛이 특징이다. 에어로프레스는 공기압을 이용해 커피를 우려내는 방법으로,
부드럽지만 깨끗한 맛이 특징이다."""
    },
    {
        "title": "커피 원두의 종류",
        "content": """커피 원두는 크게 아라비카와 로부스타로 나뉘다. 아라비카는 전 세계 커피 생산량의 약 60%를 차지하며,
산미와 단맛의 밸런스가 좋다. 로부스타는 약 40%를 차지하며, 쓴맛과 목질한 향이 특징이다.
단일 원산지 커피(Single Origin)는 한 지역의 커피만을 사용하여 그 지역의 독특한 풍미를 즐길 수 있다.
블렌드 커피는 여러 원두를 섮어 일정한 맛을 내는 방법으로, 대부분의 카페에서 사용한다."""
    },
    {
        "title": "커피와 건강",
        "content": """커피에는 카페인, 항산화물질, 클로로겐산 등 다양한 성분이 포함되어 있다.
적당량의 커피 섬취(하루 3~4잔)는 집중력 향상, 운동 능력 향상, 제2형 당뜨 위험 감소 등의
효과가 있다고 알려져 있다. 다만 과다한 카페인 섭취는 불면증, 두근거림, 소화 장애 등을
유발할 수 있으므로 주의가 필요하다. 임산부나 카페인에 민감한 사람은 섭취를 제한해야 한다."""
    }
]

print(f"문서 수: {len(documents)}")
for doc in documents:
    print(f"  - {doc['title']} ({len(doc['content'])}글자)")

---
## 3. 문서 청킹 전략

LLM의 컨텍스트 윈도우는 제한이 있으므로, 문서를 적절한 크기로 나눠야 한다.

| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **고정 크기** | N글자/토큰 단위 | 구현 간단 | 문장 중간에서 잘림 |
| **문장 단위** | 문장 경계로 분할 | 의미 보존 | 문장 길이 다양 |
| **의미 단위** | 단락/섹션 단위 | 상황 보존 | 구현 복잡 |
| **오버랩** | 청크 간 중복 구간 | 컨텍스트 유지 | 저장량 증가 |

In [ ]:
# --- 청킹 전략 구현 ---

def chunk_fixed_size(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    """고정 글자 수 단위 청킹 (overlap 포함)"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


def chunk_by_sentence(text: str, max_sentences: int = 3) -> list[str]:
    """문장 단위 청킹"""
    # 한국어 문장 분리 (\uac04\ub2e8 \ubc84\uc804)
    sentences = re.split(r'(?<=[.\ub2e4.\uc694.\uc74c.\uc2b5\ub2c8\ub2e4.])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences])
        chunks.append(chunk)
    return chunks


def chunk_by_paragraph(text: str) -> list[str]:
    """단락 단위 청킹"""
    paragraphs = text.split("\n")
    return [p.strip() for p in paragraphs if p.strip()]


# 청킹 비교
sample_text = documents[0]["content"]
print("=== 원문 ===")
print(sample_text[:100] + "...")

print("\n=== 고정 크기 (100글자, overlap 20) ===")
fixed_chunks = chunk_fixed_size(sample_text, 100, 20)
for i, chunk in enumerate(fixed_chunks):
    print(f"  Chunk {i}: [{len(chunk)}글자] {chunk[:50]}...")

print("\n=== 문장 단위 (3문장씩) ===")
sent_chunks = chunk_by_sentence(sample_text, 3)
for i, chunk in enumerate(sent_chunks):
    print(f"  Chunk {i}: [{len(chunk)}글자] {chunk[:50]}...")

In [ ]:
# 전체 문서 청킹

all_chunks = []
chunk_metadata = []

for doc in documents:
    chunks = chunk_by_sentence(doc["content"], max_sentences=2)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "doc_title": doc["title"],
            "chunk_index": i,
            "text": chunk
        })

print(f"전체 청크 수: {len(all_chunks)}")
print(f"\ud3c9\uade0 \uccad\ud06c \uae38\uc774: {np.mean([len(c) for c in all_chunks]):.0f}\uae00\uc790")
print(f"\ucd5c\uc18c: {min(len(c) for c in all_chunks)}, \ucd5c\ub300: {max(len(c) for c in all_chunks)}")
print("\n\ucc98\uc74c 5\uac1c \uccad\ud06c:")
for i, meta in enumerate(chunk_metadata[:5]):
    print(f"  [{meta['doc_title']}] {meta['text'][:50]}...")

---
## 4. Embedding 모델

텍스트를 벡터로 변환하여 의미적 유사도를 계산할 수 있게 한다.

- **sentence-transformers**: 로컸에서 실행 가능한 임베딩 모델
- 유사한 의미의 텍스트 → 가까운 벡터
- 다른 의미의 텍스트 → 먼 벡터

$$\text{similarity}(q, d) = \cos(\theta) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \|\mathbf{d}\|}$$

In [ ]:
# --- Embedding 생성 ---

if SBERT_AVAILABLE:
    # 실제 sentence-transformers 사용
    model = SentenceTransformer('all-MiniLM-L6-v2')  # 가벼운 모델
    embeddings = model.encode(all_chunks, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    print(f"Embedding shape: {embeddings.shape}")
    print(f"\u2192 {len(all_chunks)}개 청크 × {embeddings.shape[1]}차원 벡터")
else:
    # Mock embedding: TF-IDF 스타일의 간단한 벡터화
    print("SentenceTransformers 미설치 - 간단한 해시 기반 임베딩 사용")
    
    # 간단한 해시 기반 임베딩 (교육용)
    np.random.seed(42)
    EMBED_DIM = 128
    
    # 키워드 기반으로 유사한 텍스트는 유사한 벡터를 가지도록 설계
    keyword_vectors = {}
    keywords = ["커피", "카페", "성수동", "블루보틀", "스타벅스", "로스터리",
               "드립", "원두", "아라비카", "건강", "카페인", "핸드드립",
               "에스프레소", "콜드브루", "서울", "홍대", "연남동"]
    for kw in keywords:
        keyword_vectors[kw] = np.random.randn(EMBED_DIM).astype('float32')
    
    def simple_embed(text: str) -> np.ndarray:
        """\ud0a4\uc6cc\ub4dc \uae30\ubc18 \uac04\ub2e8 \uc784\ubca0\ub529"""
        vec = np.zeros(EMBED_DIM, dtype='float32')
        count = 0
        for kw, kv in keyword_vectors.items():
            if kw in text:
                vec += kv
                count += 1
        if count > 0:
            vec /= count
        # 정\uaddch\ud654
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec
    
    embeddings = np.array([simple_embed(chunk) for chunk in all_chunks]).astype('float32')
    print(f"Embedding shape: {embeddings.shape}")
    print(f"\u2192 {len(all_chunks)}\uac1c \uccad\ud06c \u00d7 {embeddings.shape[1]}\ucc28\uc6d0 \ubca1\ud130")

---
## 5. 벡터 DB: FAISS

FAISS (Facebook AI Similarity Search)는 대규모 벡터 검색을 효율적으로 수행하는 라이브러리.

### 주요 인덱스 타입

| 인덱스 | 설명 | 용도 |
|---------|------|------|
| **IndexFlatL2** | 정확한 L2 거리 검색 | 소규모 (<10만 건) |
| **IndexFlatIP** | 내적 (Inner Product) 검색 | 코사인 유사도 |
| **IndexIVFFlat** | 클러스터링 + 검색 | 중규모 |
| **IndexHNSW** | 그래프 기반 근사 검색 | 대규모, 매우 빠름 |

In [ ]:
# --- 벡터 DB 구축 및 검색 ---

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


if FAISS_AVAILABLE:
    # FAISS 인덱스 생성
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product (\ucf54\uc0ac\uc778 \uc720\uc0ac\ub3c4)
    
    # 정규h\ud654 \ud6c4 \ucd94\uac00 (\ucf54\uc0ac\uc778 \uc720\uc0ac\ub3c4\uc6a9)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)
    print(f"FAISS \uc778\ub371\uc2a4 \uc0dd\uc131 \uc644\ub8cc: {index.ntotal}\uac1c \ubca1\ud130")
    
    def search(query: str, k: int = 3) -> list[dict]:
        """\ucffc\ub9ac\ub85c \uac00\uc7a5 \uc720\uc0ac\ud55c \uccad\ud06c \uac80\uc0c9"""
        if SBERT_AVAILABLE:
            q_vec = model.encode([query]).astype('float32')
        else:
            q_vec = np.array([simple_embed(query)]).astype('float32')
        faiss.normalize_L2(q_vec)
        
        scores, indices = index.search(q_vec, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append({
                "chunk": all_chunks[idx],
                "score": float(score),
                "metadata": chunk_metadata[idx]
            })
        return results

else:
    # NumPy 기반 검색 (FAISS 없을 때)
    print("NumPy \uae30\ubc18 \ubca1\ud130 \uac80\uc0c9 \uc0ac\uc6a9")
    
    def search(query: str, k: int = 3) -> list[dict]:
        """NumPy\ub85c \ucf54\uc0ac\uc778 \uc720\uc0ac\ub3c4 \uac80\uc0c9"""
        if SBERT_AVAILABLE:
            q_vec = model.encode([query])[0]
        else:
            q_vec = simple_embed(query)
        
        scores = [cosine_similarity(q_vec, emb) for emb in embeddings]
        top_indices = np.argsort(scores)[::-1][:k]
        
        results = []
        for idx in top_indices:
            results.append({
                "chunk": all_chunks[idx],
                "score": float(scores[idx]),
                "metadata": chunk_metadata[idx]
            })
        return results


# 검색 테스트
test_queries = [
    "성수동에 유명한 카페는?",
    "카페인이 건강에 좋은가요?",
    "커피 드립 방법 종류는?",
]

for query in test_queries:
    print(f"\n=== Query: \"{query}\" ===")
    results = search(query, k=3)
    for i, r in enumerate(results):
        print(f"  #{i+1} [score={r['score']:.3f}] [{r['metadata']['doc_title']}]")
        print(f"      {r['chunk'][:60]}...")

---
## 6. 전체 RAG 파이프라인 구현

검색된 청크를 컨텍스트로 주어 LLM이 답변을 생성하도록 한다.

In [ ]:
# --- RAG 파이프라인 ---

def mock_llm_generate(prompt: str) -> str:
    """
    Mock LLM: 컨텍스트 기반으로 답변 생성 시뮬레이션.
    실제로는 OpenAI API 등을 호출.
    """
    prompt_lower = prompt.lower()
    
    if "성수동" in prompt and "카페" in prompt:
        return ("제공된 문서에 따르면, 성수동은 트렌디한 카페와 로스터리가 밀집해 있는 지역입니다. "
                "블루보틀은 2019년에 개점하여 스페셜티 드립 커피로 유명해졌으며, "
                "컨테이너 등 다양한 카페가 위치해 있습니다.")
    elif "건강" in prompt or "카페인" in prompt:
        return ("제공된 문서에 따르면, 적당량의 커피 섬취(하루 3~4잔)는 집중력 향상, "
                "운동 능력 향상, 제2형 당뜨 위험 감소 등의 효과가 있습니다. "
                "다만 과다한 카페인 섭취는 불면증 등을 유발할 수 있으므로 주의가 필요합니다.")
    elif "드립" in prompt or "방법" in prompt:
        return ("제공된 문서에 따르면, 주요 커피 드립 방법으로는 "
                "핸드드립(뜨거운 물로 천천히), 에스프레소(고압 추출), "
                "콜드브루(차가운 물로 장시간), 에어로프레스(공기압) 등이 있습니다.")
    else:
        return "제공된 문서에서 관련 정보를 찾을 수 없습니다."


def rag_pipeline(query: str, k: int = 3) -> dict:
    """
    완전한 RAG 파이프라인.
    1. 쿼리 임베딩
    2. 벡터 DB 검색
    3. 컨텍스트 구성
    4. LLM 생성
    """
    # Step 1-2: 검색
    retrieved = search(query, k=k)
    
    # Step 3: 컨텍스트 구성
    context = "\n\n".join([f"[\ubb38\uc11c: {r['metadata']['doc_title']}]\n{r['chunk']}" 
                           for r in retrieved])
    
    prompt = f"""\ub2e4\uc74c \ubb38\uc11c\ub97c \ucc38\uace0\ud558\uc5ec \uc9c8\ubb38\uc5d0 \ub2f5\ud558\uc138\uc694. \ubb38\uc11c\uc5d0 \uc5c6\ub294 \ub0b4\uc6a9\uc740 \ub2f5\ud558\uc9c0 \ub9c8\uc138\uc694.

## \ucc38\uace0 \ubb38\uc11c
{context}

## \uc9c8\ubb38
{query}

## \ub2f5\ubcc0"""
    
    # Step 4: 생성
    answer = mock_llm_generate(prompt)
    
    return {
        "query": query,
        "retrieved": retrieved,
        "context": context,
        "prompt": prompt,
        "answer": answer
    }


# RAG 테스트
test_queries = [
    "성수동에 유명한 카페는 어디인가요?",
    "커피가 건강에 좋은가요?",
    "커피 드립 방법에는 어떤 것들이 있나요?"
]

for query in test_queries:
    result = rag_pipeline(query)
    print(f"\n{'='*60}")
    print(f"Q: {result['query']}")
    print(f"\n검색된 문서 ({len(result['retrieved'])}개):")
    for i, r in enumerate(result['retrieved']):
        print(f"  #{i+1} [{r['metadata']['doc_title']}] score={r['score']:.3f}")
    print(f"\nA: {result['answer']}")

---
## 7. RAG 평가: Faithfulness, Relevancy

### RAG 평가 지표

| 지표 | 설명 | 측정 방법 |
|------|------|----------|
| **Faithfulness** | 답변이 검색된 문서에 근거하는가 | 답변의 각 주장이 컨텍스트에 있는지 확인 |
| **Context Relevancy** | 검색된 문서가 질문과 관련 있는가 | 쿼리-청크 유사도 |
| **Answer Relevancy** | 답변이 질문에 적절한가 | 답변-질문 유사도 |

In [ ]:
# --- 간단한 RAG 평가 구현 ---

def evaluate_faithfulness(answer: str, context: str) -> float:
    """
    답변이 컨텍스트에 근거하는지 평가.
    간단 버전: 답변의 키워드가 컨텍스트에 얼마나 존재하는지.
    """
    # 답변에서 주요 언급 추출 (명사 위주)
    answer_keywords = set(re.findall(r'[\uac00-\ud7a3]{2,}', answer))
    context_text = context.lower()
    
    if not answer_keywords:
        return 0.0
    
    grounded_count = sum(1 for kw in answer_keywords if kw in context_text)
    return grounded_count / len(answer_keywords)


def evaluate_context_relevancy(query: str, retrieved: list[dict]) -> float:
    """검색된 문서의 질문 대비 관련성 평가."""
    if not retrieved:
        return 0.0
    avg_score = np.mean([r["score"] for r in retrieved])
    return float(max(0, min(1, avg_score)))


def evaluate_answer_relevancy(query: str, answer: str) -> float:
    """답변이 질문에 적절한지 평가."""
    query_keywords = set(re.findall(r'[\uac00-\ud7a3]{2,}', query))
    if not query_keywords:
        return 0.0
    
    covered = sum(1 for kw in query_keywords if kw in answer)
    return covered / len(query_keywords)


# 평가 실행
eval_results = []

for query in test_queries:
    result = rag_pipeline(query)
    
    faithfulness = evaluate_faithfulness(result["answer"], result["context"])
    ctx_relevancy = evaluate_context_relevancy(query, result["retrieved"])
    ans_relevancy = evaluate_answer_relevancy(query, result["answer"])
    
    eval_results.append({
        "query": query[:30],
        "faithfulness": faithfulness,
        "context_relevancy": ctx_relevancy,
        "answer_relevancy": ans_relevancy,
    })

eval_df = pd.DataFrame(eval_results)
print("=== RAG \ud3c9\uac00 \uacb0\uacfc ===")
print(eval_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(eval_df))
width = 0.25

ax.bar(x - width, eval_df["faithfulness"], width, label="Faithfulness", color="#4ecdc4")
ax.bar(x, eval_df["context_relevancy"], width, label="Context Relevancy", color="#45b7d1")
ax.bar(x + width, eval_df["answer_relevancy"], width, label="Answer Relevancy", color="#f7dc6f")

ax.set_xlabel("Query")
ax.set_ylabel("Score")
ax.set_title("RAG Evaluation Metrics")
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in range(len(eval_df))])
ax.legend()
ax.set_ylim(0, 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 청킹 전략 비교

동일한 문서에 대해 3가지 청킹 전략을 적용하고, 각각의 검색 품질을 비교하세요.

- 고정 크기 (50글자, overlap 10)
- 문장 단위 (2문장)
- 문장 단위 (4문장)

In [ ]:
# TODO: 3가지 청킹 전략을 적용하고 검색 품질을 비교하세요
# 요구사항:
# 1. 각 전략으로 청킹
# 2. 임베딩 생성 + 벡터 DB 구축
# 3. 동일한 쿼리로 검색
# 4. 검색 점수 비교


### 연습 2: RAG 파이프라인 확장

새로운 문서를 추가하고 RAG 파이프라인을 테스트하세요.

- 새 문서: 자신이 아는 커피 지식을 3개의 문단으로 작성
- 새 문서를 청킹 → 임베딩 → 벡터 DB에 추가
- 새 문서 관련 질문으로 테스트

In [ ]:
# TODO: 새 문서를 추가하고 RAG 파이프라인을 테스트하세요
# new_doc = {
#     "title": "...",
#     "content": "..."
# }


---
## 핵심 정리

| 개념 | 설명 | 활용 |
|------|------|------|
| RAG | 검색 + 생성으로 hallucination 감소 | 지식 기반 Q&A |
| 청킹 | 문서를 적절한 크기로 분할 | 처리 단위 결정 |
| Embedding | 텍스트 → 벡터 변환 | 의미 유사도 계산 |
| FAISS | 대규모 벡터 검색 | 빠른 유사 문서 검색 |
| Faithfulness | 답변이 근거에 기반하는가 | RAG 품질 평가 |
| Context Relevancy | 검색 결과가 관련 있는가 | 검색 품질 평가 |

**다음 노트북**: [05-evaluation-pipeline.ipynb](05-evaluation-pipeline.ipynb) - 환각 탐지 & 평가 파이프라인